# 1 Os dados foram retirados da fonte [Aneel](https://dadosabertos.aneel.gov.br/dataset/interrupcoes-de-energia-eletrica-nas-redes-de-distribuicao) do ano de 2025


In [6]:
import pandas as pd
import json

# ========================================
# CONFIGURAÇÕES
# ========================================

CSV_FILE = "interrupcoes-energia-eletrica-2025.csv"
OUTPUT_FILE = "dataset.jsonl"

# Escolha:
# 1000
# 50000
# 100000
NUM_INSTANCIAS = 1000

SHUFFLE = True

# ========================================
# LEITURA DO CSV
# ========================================

df = pd.read_csv(
    CSV_FILE,
    sep=";",
    encoding="latin1"
)

# Converte datas
df["DatInicioInterrupcao"] = pd.to_datetime(df["DatInicioInterrupcao"])
df["DatFimInterrupcao"] = pd.to_datetime(df["DatFimInterrupcao"])

# ========================================
# AUMENTA O DATASET (caso necessário)
# ========================================

if len(df) >= NUM_INSTANCIAS:
    df = df.sample(NUM_INSTANCIAS, random_state=42)

else:
    repeticoes = (NUM_INSTANCIAS // len(df)) + 1

    df = pd.concat([df] * repeticoes, ignore_index=True)

    if SHUFFLE:
        df = df.sample(frac=1, random_state=42)

    df = df.iloc[:NUM_INSTANCIAS]

df = df.reset_index(drop=True)

# ========================================
# GERAÇÃO DOS EVENTOS
# ========================================

eventos = []

for indice, linha in df.iterrows():

    # Duração da interrupção em minutos
    duracao = (
        linha["DatFimInterrupcao"] -
        linha["DatInicioInterrupcao"]
    ).total_seconds() / 60

    # Gravidade baseada na duração
    if duracao < 30:
        gravidade = 1
    elif duracao < 60:
        gravidade = 2
    elif duracao < 120:
        gravidade = 3
    elif duracao < 240:
        gravidade = 4
    else:
        gravidade = 5

    # Simplifica a descrição
    descricao = (
        str(linha["DscFatoGeradorInterrupcao"])
        .replace("INTERNA - ", "")
        .replace("EXTERNA - ", "")
        .replace("NAO PROGRAMADA - ", "")
        .replace("PROGRAMADA - ", "")
        .replace("PROPRIAS DO SISTEMA - ", "")
        .replace("_", " ")
        .title()
    )

    evento = {

        "idEvento": f"EVT{indice + 1:06}",

        "descricao": descricao,

        "dataHoraInicio": linha["DatInicioInterrupcao"].isoformat(),

        "dataHoraFim": linha["DatFimInterrupcao"].isoformat(),

        "duracaoMinutos": round(duracao, 2),

        "gravidade": gravidade,

        "conjuntoConsumidor": str(
            linha["DscConjuntoUnidadeConsumidora"]
        ).title(),

        "alimentador": str(
            linha["DscAlimentadorSubestacao"]
        ).strip(),

        "subestacao": str(
            linha["DscSubestacaoDistribuicao"]
        ).strip(),

        "tipoInterrupcao": str(
            linha["DscTipoInterrupcao"]
        ).strip(),

        "agenteRegulado": str(
            linha["NomAgenteRegulado"]
        ).strip(),

        "siglaAgente": str(
            linha["SigAgente"]
        ).strip()

    }

    eventos.append(evento)


# ========================================
# EXPORTAÇÃO JSONL
# ========================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for evento in eventos:
        f.write(json.dumps(evento, ensure_ascii=False))
        f.write("\n")

print(f"Arquivo '{OUTPUT_FILE}' gerado com {len(eventos)} eventos.")

/tmp/ipykernel_178515/4159910969.py:23: DtypeWarning: Columns (0: DscAlimentadorSubestacao, 1: NumOrdemInterrupcao) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Arquivo 'dataset.jsonl' gerado com 1000 eventos.
